# Pretty Versus Practical: Response to a Classmate
Margaret Gonzalez

Word cloud of the unigrams from the classmate's analysis of the NIST AI Risk Management Framework (AI RMF 1.0),
printed pages 1 to 19 (PDF pages 6 to 24). The text preparation below is the classmate's own code, reused
unchanged, so the cloud is built from exactly the same counts as his frequency plots.

Files needed in the same folder: `NIST_AI_100-1.pdf` (from his post) and `shield_mask.png` (created by
cell 3 if it is missing, and attached to my post).

In [ ]:
%pip install -q pypdf scikit-learn wordcloud matplotlib pillow numpy

## 1. The classmate's preparation code, reused as-is

In [ ]:
from pathlib import Path
from collections import Counter
import re, unicodedata
from pypdf import PdfReader
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw
from wordcloud import WordCloud

source = Path("NIST_AI_100-1.pdf")          # relative path
reader = PdfReader(source)
selected = reader.pages[5:24]               # PDF pages 6-24, printed pages 1-19
stopwords = set(ENGLISH_STOP_WORDS) | {"s", "t"}

unigrams = Counter()
for page in selected:
    text = unicodedata.normalize("NFKC", page.extract_text() or "")
    text = re.sub(r"(?m)^NIST AI 100-1\s+AI RMF 1\.0\s*$", "", text)
    text = re.sub(r"(?m)^Page\s+\d+\s*$", "", text)
    text = re.sub(r"(?<=[A-Za-z])-\s*\n\s*(?=[a-z])", "", text)
    text = re.sub(r"https?://\S+", " ", text)
    text = text.lower()
    for segment in re.split(r"[.!?;:]", text):
        tokens = re.findall(r"[a-z]+", segment)
        unigrams.update(w for w in tokens if len(w) > 1 and w not in stopwords)

print("PDF pages:", len(reader.pages), "| pages analyzed:", len(selected))
print("Retained unigram occurrences:", sum(unigrams.values()))
print("Distinct unigrams:", len(unigrams))
print("Top 10:", unigrams.most_common(10))

**Check against the classmate's post.** He reported 3,977 retained occurrences, 1,228 distinct terms, and
`ai` 289, `risk` 95, `systems` 94, `risks` 67. The output above should match those numbers exactly,
which confirms the cloud is built from his data and not from a different cleaning pass.

## 2. One thing I noticed in the stopword list

In [ ]:
all_tokens = Counter()
for page in selected:
    text = unicodedata.normalize("NFKC", page.extract_text() or "")
    text = re.sub(r"(?m)^NIST AI 100-1\s+AI RMF 1\.0\s*$", "", text)
    text = re.sub(r"(?m)^Page\s+\d+\s*$", "", text)
    text = re.sub(r"(?<=[A-Za-z])-\s*\n\s*(?=[a-z])", "", text)
    text = re.sub(r"https?://\S+", " ", text).lower()
    all_tokens.update(re.findall(r"[a-z]+", text))

print("Is 'system' in scikit-learn's ENGLISH_STOP_WORDS?", "system" in ENGLISH_STOP_WORDS)
print("Is 'systems' in it?", "systems" in ENGLISH_STOP_WORDS)
print("'system' occurrences removed as a stopword:", all_tokens["system"])
print("'systems' occurrences kept:", all_tokens["systems"])
print("Combined:", all_tokens["system"] + all_tokens["systems"])

## 3. The shape image

In [ ]:
mask_path = Path("shield_mask.png")
if not mask_path.exists():
    W, H = 1000, 1150
    img = Image.new("L", (W, H), 255)                    # white = no words
    d = ImageDraw.Draw(img)
    top = [(80, 120), (500, 30), (920, 120)]             # shield top edge
    left = [(80, 120), (80, 520), (130, 760), (270, 960), (500, 1110)]
    right = [(920, 120), (920, 520), (870, 760), (730, 960), (500, 1110)]
    outline = [top[0], top[1], top[2]] + right[1:] + left[::-1][1:-1]
    d.polygon(outline, fill=0)                           # black = words go here
    img.save(mask_path)
mask = np.array(Image.open(mask_path).convert("L"))
mask = np.where(mask > 128, 255, 0).astype(np.uint8)     # clean binary mask
print("Mask size:", mask.shape, "| shape pixels:", int((mask == 0).sum()))
plt.figure(figsize=(3, 3.4)); plt.imshow(mask, cmap="gray"); plt.axis("off"); plt.title("shield_mask.png"); plt.show()

## 4. The word cloud

In [ ]:
PALETTE = ["#F2B134", "#4FB0C6", "#E8EEF2", "#8FD694"]    # amber, teal, off white, green
ranked = [w for w, _ in unigrams.most_common()]
tier = {w: (0 if i < 5 else 1 if i < 25 else 2 if i < 60 else 3) for i, w in enumerate(ranked)}

def tier_color(word, **kwargs):
    return PALETTE[tier.get(word, 3)]

# Rebuild the text from the classmate's retained tokens so collocations=False has real text to act on
cloud_text = " ".join(w for w, n in unigrams.items() for _ in range(n))

import os
FONT_CANDIDATES = ["C:/Windows/Fonts/arialbd.ttf",
                   "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"]
font_path = next((f for f in FONT_CANDIDATES if os.path.exists(f)), None)
print("Font:", font_path or "wordcloud default")

wc = WordCloud(
    font_path=font_path,             # bold sans serif (Arial Bold on Windows)
    mask=mask,                       # (a) shield shape
    background_color="#0B1F33",      # (b) deep navy background
    color_func=tier_color,           # (b) word colors by frequency tier
    collocations=False,              # required: single words only
    contour_width=4,                 # extra 1: shield outline
    contour_color="#F2B134",
    max_words=120,                   # extra 2: cap at the top 120 terms
    stopwords=set(),                 # The classmate's stopwords were already applied
    normalize_plurals=False,         # keep risk and risks separate, as in his plots
    regexp=r"[a-z]+",
    min_font_size=8,
    margin=6,
    random_state=7,
).generate(cloud_text)

plt.figure(figsize=(9, 10.35))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("NIST AI RMF 1.0: Top 120 Unigrams", fontsize=16, pad=12)
plt.tight_layout()
plt.savefig("nist_wordcloud.png", dpi=150, bbox_inches="tight")
plt.show()

print("Words drawn:", len(wc.layout_))
print("Largest five:", [w for (w, _), *_ in wc.layout_[:5]])

## 5. Customization choices

| Element | Setting | Reason |
|---|---|---|
| Shape (a) | `shield_mask.png` | The framework is about protecting people and organizations from AI harm, and a shield reads as protection at a glance. |
| Background (b) | Navy `#0B1F33` | Dark, neutral and close to the government report look. Every word color passes easily against it. |
| Word colors (b) | Amber for the top 5, teal for 6 to 25, off white for 26 to 60, green for the rest | Color adds a second cue for rank, so `ai`, `risk`, `systems`, `risks` and `management` separate from the pack even where font sizes are close. |
| Extra 1 | `contour_width=4`, amber | Without an outline, a filled shield on a dark background loses its edge and the shape is hard to recognize. |
| Extra 2 | `max_words=120` | The default 200 pushes in dozens of words used two or three times, which become unreadable specks. 120 keeps the cloud legible. |
| Supporting | Bold sans serif font, `margin=6` | The default font draws the dot of a huge `i` far from its stem, so the dot of `ai` was landing beside other words and reading like a period. A bold font plus extra padding keeps each word visually separate. |
| Required | `collocations=False` | Stops pairs like `ai systems` from being drawn as a single term, so the cloud shows unigrams only. |

## References

[1] National Institute of Standards and Technology, *Artificial Intelligence Risk Management Framework
(AI RMF 1.0)*, NIST AI 100-1, Jan. 2023. [Online]. Available: https://doi.org/10.6028/NIST.AI.100-1

[2] A. Mueller, "wordcloud: A little word cloud generator in Python." [Online]. Available:
https://amueller.github.io/word_cloud/

[3] scikit-learn developers, "sklearn.feature_extraction.text.ENGLISH_STOP_WORDS." [Online]. Available:
https://github.com/scikit-learn/scikit-learn/blob/main/sklearn/feature_extraction/_stop_words.py

[4] J. Nothman, H. Qin, and R. Yurchak, "Stop word lists in free open-source software packages," in
*Proc. Workshop for NLP Open Source Software (NLP-OSS)*, 2018, pp. 7-12. [Online]. Available:
https://aclanthology.org/W18-2502/